# Variant D — Hugging Face API e-SNLI-Guided CoT/Rationale Generation (Part 2)

This notebook generates part of the **2,000 total** balanced e-SNLI rationale traces for **Model D** using the **Hugging Face Inference Providers OpenAI-compatible API**.

Default generator: `deepseek-ai/DeepSeek-R1-Distill-Qwen-7B` through Hugging Face. This model is only the **teacher/generator** for rationales. Your actual Model D classifier can still use the same backbone as Models A/B/C.

Split plan:

- **Part 1 notebook:** generates rows `pair_id` 0–999 → `cot_traces_part1.csv`
- **Part 2 notebook:** generates rows `pair_id` 1000–1999 → `cot_traces_part2.csv`
- The final concatenated file is `cot_traces.csv`.

The generated CSV supports the Model D sub-configurations:

- **D-Human:** `model_d_human_text`
- **D-CoT:** `model_d_cot_text`
- **D-Blend:** `model_d_blend_text`

Everything is saved to Google Drive and can resume after a Colab timeout.


In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
# Hugging Face API version: no local model server or system install needed.
!pip install -q pandas==2.2.2 requests tqdm


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  Cell 2: Configuration
# ══════════════════════════════════════════════════════════════════════

# Edit this to your Google Drive project folder.
PROJECT_ROOT = "/content/drive/MyDrive/Old-Explanations-New-Rationales-Bridging-e-SNLI-and-LLM-Chain-of-Thought-in-Encoder-Based-NLI"

# Hugging Face Inference Providers OpenAI-compatible router.
# Put your token in Colab Secrets as HF_TOKEN, or set os.environ["HF_TOKEN"].
API_BASE_URL = "https://router.huggingface.co/v1"
API_CHAT_URL = f"{API_BASE_URL}/chat/completions"
API_KEY_ENV_VAR = "HF_TOKEN"
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
MODEL_PROVIDER = "Hugging Face Inference Providers"

# Full experiment target. The two notebooks split this into 1,000 + 1,000.
TOTAL_TRACES = 2000

# This notebook's assigned slice.
PART_NUMBER = 2
PART_START = 1000
PART_END_EXCLUSIVE = 2000
PART_NAME = f"part{PART_NUMBER}"

# Sampling seed makes the shared subset reproducible.
RANDOM_SEED = 42

# Generation settings.
MAX_WORDS = 100
MAX_RETRIES_PER_ROW = 3
REQUEST_TIMEOUT_SECONDS = 120
TEMPERATURE = 0.2
TOP_P = 0.9
MAX_TOKENS = 260

import os
from pathlib import Path

ESNLI_TRAIN_1 = os.path.join(PROJECT_ROOT, "Datasets", "E-SNLI", "esnli_train_1.csv")
ESNLI_TRAIN_2 = os.path.join(PROJECT_ROOT, "Datasets", "E-SNLI", "esnli_train_2.csv")

COT_DIR = os.path.join(PROJECT_ROOT, "Datasets", "CoT")
SUBSET_PATH = os.path.join(COT_DIR, "cot_subset_2000.csv")

# Part-specific files prevent the two notebooks from overwriting each other.
OUTPUT_PATH = os.path.join(COT_DIR, f"cot_traces_{PART_NAME}.csv")
CHECKPOINT_PATH = os.path.join(COT_DIR, f"cot_generation_checkpoint_{PART_NAME}.json")

# Final merged files. The merge cell is included at the end of Part 2.
PART1_OUTPUT_PATH = os.path.join(COT_DIR, "cot_traces_part1.csv")
PART2_OUTPUT_PATH = os.path.join(COT_DIR, "cot_traces_part2.csv")
FINAL_OUTPUT_PATH = os.path.join(COT_DIR, "cot_traces.csv")
FINAL_DEDUPED_PATH = os.path.join(COT_DIR, "cot_traces_deduped.csv")

print(f"Project root: {PROJECT_ROOT}")
print(f"Provider:     {MODEL_PROVIDER}")
print(f"Model:        {MODEL_ID}")
print(f"Full target:  {TOTAL_TRACES:,} traces")
print(f"This part:    {PART_NAME} rows {PART_START}–{PART_END_EXCLUSIVE - 1}")
print(f"Part output:  {OUTPUT_PATH}")
print(f"Checkpoint:   {CHECKPOINT_PATH}")


In [ ]:
# ── Cell 3: Mount Google Drive and verify data paths ──────────────────────
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

Path(COT_DIR).mkdir(parents=True, exist_ok=True)

required_paths = [ESNLI_TRAIN_1, ESNLI_TRAIN_2]
missing = [p for p in required_paths if not Path(p).exists()]

for p in required_paths:
    print(("✓" if Path(p).exists() else "✗"), p)

if missing:
    raise FileNotFoundError(
        "Missing e-SNLI CSV file(s). Check PROJECT_ROOT in Cell 2:\n"
        + "\n".join(missing)
    )


In [ ]:
# ── Cell 4: Hugging Face API key setup ───────────────────────────────────
# This notebook uses the Hugging Face API only.
# Recommended in Colab: left sidebar → Secrets → add HF_TOKEN and enable notebook access.

import os

try:
    from google.colab import userdata
except Exception:
    userdata = None


def get_api_key(env_var: str = API_KEY_ENV_VAR) -> str:
    # Read the Hugging Face token from environment or Colab Secrets.
    key = os.environ.get(env_var, "").strip()

    if not key and userdata is not None:
        try:
            key = (userdata.get(env_var) or "").strip()
        except Exception:
            key = ""

    if not key:
        raise RuntimeError(
            f"Missing Hugging Face token. Add it as a Colab Secret named {env_var}, "
            f"or run: os.environ['{env_var}'] = 'hf_your_token_here' before this cell."
        )

    return key

HF_TOKEN = get_api_key()
API_KEY = HF_TOKEN
print(f"✓ Hugging Face token loaded from {API_KEY_ENV_VAR}")
print(f"✓ Using {MODEL_PROVIDER}: {MODEL_ID}")


In [ ]:
# ── Cell 5: Imports, e-SNLI-guided prompt, cleaning, and HF caller ───────
import json
import re
import time
from datetime import datetime
from pathlib import Path
from typing import Optional, Set

import pandas as pd
import requests
from tqdm.auto import tqdm

VALID_LABELS = {"entailment", "neutral", "contradiction"}

SYSTEM_PROMPT = """You generate concise e-SNLI-style natural-language rationales for NLI training data.
Return only the requested text. Do not include hidden reasoning tags, markdown headings, or <think> blocks."""

PROMPT_TEMPLATE = """You are generating a rationale for an NLI example using the e-SNLI annotation methodology, with a modern CoT-style structure.

The original e-SNLI setup gives annotators the premise, hypothesis, and gold label. The explanation should justify why the label is correct.

Methodology rules:
- Focus on the non-obvious evidence that determines the relation.
- Do not copy the full premise or full hypothesis.
- Make the explanation self-contained.
- Do not add outside facts.
- Avoid empty templates such as "premise implies hypothesis" or "just because premise does not mean hypothesis".
- For entailment, explain what in the premise supports the hypothesis.
- For neutral, explain what information is missing or cannot be inferred.
- For contradiction, explain which facts cannot both be true.
- Use 2 to 4 short reasoning steps, then one final e-SNLI-style explanation sentence.
- Do not mention that the gold label was provided.
- Keep the total response under {max_words} words.

Premise:
{premise}

Hypothesis:
{hypothesis}

Gold label:
{label}

Output format exactly:
Reasoning:
1. ...
2. ...
3. ...

Final explanation: ..."""

BANNED_PATTERNS = [
    r"<think>", r"</think>", r"^\s*thinking",
    r"just because\s+.+\s+doesn.t\s+mean\s+.+$",
    r"^premise implies hypothesis\.?$",
    r"^the premise implies the hypothesis\.?$",
    r"^cannot infer the hypothesis\.?$",
]
BANNED_RE = [re.compile(p, re.IGNORECASE | re.DOTALL) for p in BANNED_PATTERNS]

def build_prompt(premise: str, hypothesis: str, label: str) -> str:
    label = str(label).strip().lower()
    if label not in VALID_LABELS:
        raise ValueError(f"Invalid NLI label: {label}")
    return PROMPT_TEMPLATE.format(
        premise=str(premise).strip(),
        hypothesis=str(hypothesis).strip(),
        label=label,
        max_words=MAX_WORDS,
    )

def strip_think_blocks(text: str) -> str:
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"</?think>", "", text, flags=re.IGNORECASE)
    return text.strip()

def normalize_text(text: str) -> str:
    text = strip_think_blocks(text)
    text = re.sub(r"\r\n?", "\n", text).strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    words = text.split()
    if len(words) > MAX_WORDS:
        text = " ".join(words[:MAX_WORDS]).rstrip(" ,;:")
        if not text.endswith((".", "!", "?")):
            text += "."
    return text

def is_banned(text: str) -> bool:
    return any(pattern.search(text or "") for pattern in BANNED_RE)

def call_hf(prompt: str) -> Optional[str]:
    # Call Hugging Face's OpenAI-compatible chat completions router.
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "max_tokens": MAX_TOKENS,
    }
    try:
        response = requests.post(API_CHAT_URL, headers=headers, json=payload, timeout=REQUEST_TIMEOUT_SECONDS)
        if response.status_code in {429, 500, 502, 503, 504}:
            print(f"[HF transient status {response.status_code}] {response.text[:300]}")
            return None
        response.raise_for_status()
        data = response.json()
        return data["choices"][0]["message"]["content"]
    except Exception as exc:
        print(f"[HF request failed] {exc}")
        return None

def generate_rationale(premise: str, hypothesis: str, label: str) -> Optional[str]:
    prompt = build_prompt(premise, hypothesis, label)

    best = None
    for attempt in range(1, MAX_RETRIES_PER_ROW + 1):
        raw = call_hf(prompt)
        if not raw:
            time.sleep(2 * attempt)
            continue

        cleaned = normalize_text(raw)
        if len(cleaned.split()) >= 12:
            best = cleaned

        if best and not is_banned(best):
            return best

        time.sleep(1 * attempt)

    return best


In [ ]:
# ── Cell 6: Hugging Face API smoke test before generation ────────────────
# This verifies:
# 1. The HF_TOKEN works.
# 2. The configured Hugging Face model is reachable.
# 3. The model can generate one usable e-SNLI-guided rationale.

test_text = generate_rationale(
    premise="A dog is running through a grassy field.",
    hypothesis="An animal is outside.",
    label="entailment",
)

if not test_text:
    raise RuntimeError(
        "Smoke test failed: Hugging Face returned no usable rationale. "
        "Check HF_TOKEN, MODEL_ID, account access, and rate limits."
    )

print(f"✓ {MODEL_PROVIDER} smoke test passed")
print("Generated test rationale:")
print(test_text)
print(f"Words: {len(test_text.split())}")
print(f"Banned/template issue: {'YES' if is_banned(test_text) else 'NO'}")


In [ ]:
# ── Cell 7: Load or create balanced 2,000-row e-SNLI subset ───────────────
def load_esnli() -> pd.DataFrame:
    frames = []
    for path in [ESNLI_TRAIN_1, ESNLI_TRAIN_2]:
        frame = pd.read_csv(
            path,
            usecols=["gold_label", "Sentence1", "Sentence2", "Explanation_1"],
        )
        frames.append(frame)

    df = pd.concat(frames, ignore_index=True)
    df = df[df["gold_label"].isin(VALID_LABELS)]
    df = df.dropna(subset=["Sentence1", "Sentence2", "Explanation_1"])
    return df.reset_index(drop=True)

def balanced_label_quotas(total: int, labels) -> dict:
    """Return near-balanced quotas that sum exactly to total."""
    labels = sorted(labels)
    base = total // len(labels)
    remainder = total % len(labels)
    return {label: base + (i < remainder) for i, label in enumerate(labels)}

def load_or_create_subset() -> pd.DataFrame:
    subset_path = Path(SUBSET_PATH)
    subset_path.parent.mkdir(parents=True, exist_ok=True)

    if subset_path.exists():
        subset = pd.read_csv(subset_path, dtype={"pair_id": int})
        print(f"✓ Loaded existing shared subset: {subset_path}")
    else:
        print("Creating shared balanced 2,000-row subset from e-SNLI...")
        esnli = load_esnli()
        quotas = balanced_label_quotas(TOTAL_TRACES, VALID_LABELS)

        pieces = []
        for label, quota in quotas.items():
            group = esnli[esnli["gold_label"] == label]
            if len(group) < quota:
                raise ValueError(f"Not enough rows for label {label}: need {quota}, found {len(group)}")
            pieces.append(group.sample(n=quota, random_state=RANDOM_SEED))

        subset = (
            pd.concat(pieces, ignore_index=True)
            .sample(frac=1.0, random_state=RANDOM_SEED)
            .reset_index(drop=True)
        )
        subset.insert(0, "pair_id", range(len(subset)))
        subset.to_csv(subset_path, index=False)
        print(f"✓ Saved shared subset: {subset_path}")

    if len(subset) != TOTAL_TRACES:
        raise ValueError(
            f"Expected shared subset to contain {TOTAL_TRACES} rows, found {len(subset)}. "
            f"Delete {SUBSET_PATH} and re-run this cell to recreate it."
        )

    print("Full subset label counts:")
    print(subset["gold_label"].value_counts().sort_index())
    return subset

full_subset = load_or_create_subset()

# Select only the rows assigned to this notebook.
subset = (
    full_subset[
        (full_subset["pair_id"] >= PART_START)
        & (full_subset["pair_id"] < PART_END_EXCLUSIVE)
    ]
    .sort_values("pair_id")
    .reset_index(drop=True)
)

expected_part_size = PART_END_EXCLUSIVE - PART_START
if len(subset) != expected_part_size:
    raise ValueError(f"Expected {expected_part_size} rows for {PART_NAME}, found {len(subset)}")

print(f"\n{PART_NAME} target rows: {len(subset):,}")
print(f"{PART_NAME} pair_id range: {subset['pair_id'].min()}–{subset['pair_id'].max()}")
print(f"{PART_NAME} label counts:")
print(subset["gold_label"].value_counts().sort_index())

subset.head()


In [ ]:
# ── Cell 8: Google Drive checkpoint helpers ──────────────────────────────
# Resume rule:
# 1. The output CSV in Google Drive is the source of truth.
# 2. Any row with a non-empty cot_rationale_esnli_style is skipped on the next run.
# 3. The JSON checkpoint is a human-readable progress summary.

import csv

RATIONAL_METHOD = "e-SNLI-guided CoT prompting"
RATIONAL_SOURCE = f"{MODEL_PROVIDER}:{MODEL_ID}"

REQUIRED_OUTPUT_COLUMNS = [
    "pair_id",
    "gold_label",
    "Sentence1",
    "Sentence2",
    "Explanation_1",
    "human_explanation",
    "cot_rationale_esnli_style",
    "cot_rationale",
    "model_d_human_text",
    "model_d_cot_text",
    "model_d_blend_text",
    "rationale_method",
    "rationale_source",
    "cot_model",
    "created_at",
]

def _completion_column(df: pd.DataFrame) -> str:
    if "cot_rationale_esnli_style" in df.columns:
        return "cot_rationale_esnli_style"
    if "cot_rationale" in df.columns:
        return "cot_rationale"
    raise ValueError("No rationale column found in existing output CSV.")

def read_completed_pair_ids() -> Set[int]:
    output = Path(OUTPUT_PATH)
    if not output.exists() or output.stat().st_size == 0:
        return set()

    try:
        existing = pd.read_csv(output, dtype={"pair_id": int})
    except Exception as exc:
        message = (
            f"Could not read the Drive checkpoint CSV: {OUTPUT_PATH}" + chr(10)
            + "This usually means the previous session was interrupted while writing. "
            + "Open the CSV in Drive and remove any incomplete final line, then re-run."
        )
        raise RuntimeError(message) from exc

    col = _completion_column(existing)
    completed = existing[col].fillna("").astype(str).str.strip() != ""
    return set(existing.loc[completed, "pair_id"].astype(int).tolist())

def write_checkpoint(done_count: int, total_count: int, last_pair_id: Optional[int]) -> None:
    checkpoint = {
        "model_id": MODEL_ID,
        "model_provider": MODEL_PROVIDER,
        "rationale_method": RATIONAL_METHOD,
        "part": PART_NAME,
        "part_start": int(PART_START),
        "part_end_exclusive": int(PART_END_EXCLUSIVE),
        "done_count": int(done_count),
        "total_count": int(total_count),
        "remaining_count": int(total_count - done_count),
        "last_pair_id": None if last_pair_id is None else int(last_pair_id),
        "output_path": OUTPUT_PATH,
        "checkpoint_path": CHECKPOINT_PATH,
        "saved_in_google_drive": True,
        "updated_at": datetime.utcnow().isoformat() + "Z",
    }

    checkpoint_path = Path(CHECKPOINT_PATH)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = checkpoint_path.with_suffix(".json.tmp")
    tmp_path.write_text(json.dumps(checkpoint, indent=2))
    tmp_path.replace(checkpoint_path)

def build_model_d_texts(row, rationale: Optional[str]):
    premise = str(row.Sentence1).strip()
    hypothesis = str(row.Sentence2).strip()
    human = str(row.Explanation_1).strip()
    cot = (rationale or "").strip()

    human_text = f"Premise: {premise}
Hypothesis: {hypothesis}
Human explanation: {human}"
    cot_text = f"Premise: {premise}
Hypothesis: {hypothesis}
CoT rationale: {cot}"
    blend_text = f"Premise: {premise}
Hypothesis: {hypothesis}
Human explanation: {human}
CoT rationale: {cot}"
    return human_text, cot_text, blend_text

def append_result(row, rationale: Optional[str], file_exists: bool) -> bool:
    human_text, cot_text, blend_text = build_model_d_texts(row, rationale)
    record = {
        "pair_id": int(row.pair_id),
        "gold_label": row.gold_label,
        "Sentence1": row.Sentence1,
        "Sentence2": row.Sentence2,
        "Explanation_1": row.Explanation_1,
        "human_explanation": row.Explanation_1,
        "cot_rationale_esnli_style": rationale or "",
        "cot_rationale": rationale or "",  # backward-compatible alias
        "model_d_human_text": human_text,
        "model_d_cot_text": cot_text,
        "model_d_blend_text": blend_text,
        "rationale_method": RATIONAL_METHOD,
        "rationale_source": RATIONAL_SOURCE,
        "cot_model": MODEL_ID,
        "created_at": datetime.utcnow().isoformat() + "Z",
    }

    output_path = Path(OUTPUT_PATH)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Write one row at a time directly to the Drive-backed CSV.
    # This makes the run resumable even if Colab disconnects mid-notebook.
    with output_path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=REQUIRED_OUTPUT_COLUMNS)
        if not file_exists or output_path.stat().st_size == 0:
            writer.writeheader()
        writer.writerow(record)
        f.flush()
        os.fsync(f.fileno())

    return True

def show_resume_status(source_df: pd.DataFrame):
    done = read_completed_pair_ids()
    remaining = source_df[~source_df["pair_id"].isin(done)].reset_index(drop=True)
    print(f"Total target: {len(source_df):,}")
    print(f"Completed:    {len(done):,}")
    print(f"Remaining:    {len(remaining):,}")
    print(f"Output:       {OUTPUT_PATH}")
    print(f"Checkpoint:   {CHECKPOINT_PATH}")
    return done, remaining

done_ids, remaining = show_resume_status(subset)


In [ ]:
# ── Cell 9: Generate traces with row-level checkpoints ────────────────────
# Safe to re-run. It skips rows already present in this part's output CSV.

def generate_all(source_df: pd.DataFrame) -> None:
    done, remaining = show_resume_status(source_df)
    total = len(source_df)

    if remaining.empty:
        write_checkpoint(done_count=len(done), total_count=total, last_pair_id=None)
        print(f"✓ Nothing to do. All {PART_NAME} traces are already generated.")
        return

    output = Path(OUTPUT_PATH)
    output.parent.mkdir(parents=True, exist_ok=True)
    file_exists = output.exists()
    start = time.time()
    last_pair_id = None

    progress = tqdm(remaining.itertuples(index=False), total=len(remaining), desc=f"Generating {PART_NAME}")
    for row in progress:
        rationale = generate_rationale(row.Sentence1, row.Sentence2, row.gold_label)
        append_result(row, rationale, file_exists=file_exists)
        file_exists = True

        done.add(int(row.pair_id))
        last_pair_id = int(row.pair_id)
        write_checkpoint(done_count=len(done), total_count=total, last_pair_id=last_pair_id)

        elapsed_min = max((time.time() - start) / 60, 1e-6)
        rate = (len(done) / elapsed_min)
        progress.set_postfix(done=len(done), total=total, rate_per_min=f"{rate:.2f}")

    print(f"✓ {PART_NAME} generation finished or current run completed. Saved to {OUTPUT_PATH}")

generate_all(subset)


In [ ]:
# ── Cell 10: Validate this part's output ──────────────────────────────────
def validate_output() -> pd.DataFrame:
    output = Path(OUTPUT_PATH)
    if not output.exists():
        raise FileNotFoundError(f"No output file found yet: {OUTPUT_PATH}")

    df = pd.read_csv(output, dtype={"pair_id": int})
    rationale_col = "cot_rationale_esnli_style" if "cot_rationale_esnli_style" in df.columns else "cot_rationale"
    df = df.drop_duplicates(subset=["pair_id"], keep="last").sort_values("pair_id").reset_index(drop=True)

    expected_ids = set(range(PART_START, PART_END_EXCLUSIVE))
    actual_ids = set(df["pair_id"].astype(int).tolist())
    outside_ids = sorted(actual_ids - expected_ids)

    filled_mask = df[rationale_col].fillna("").astype(str).str.strip() != ""
    filled = int(filled_mask.sum())
    target = PART_END_EXCLUSIVE - PART_START

    label_counts = df.loc[filled_mask, "gold_label"].value_counts().sort_index()
    word_counts = df.loc[filled_mask, rationale_col].fillna("").astype(str).str.split().apply(len)
    banned_count = int(df.loc[filled_mask, rationale_col].apply(is_banned).sum())

    print("═" * 60)
    print(f"Validation Report — {PART_NAME}")
    print("═" * 60)
    print(f"Progress:          {filled:,} / {target:,} ({filled / target * 100:.1f}%)")
    print(f"Unique pair_ids:   {df['pair_id'].nunique():,}")
    print(f"Expected range:    {PART_START}–{PART_END_EXCLUSIVE - 1}")
    print(f"Outside range IDs: {len(outside_ids):,}")
    print(f"Banned/leakage:    {banned_count:,} ({banned_count / max(filled, 1) * 100:.1f}%)")
    if filled:
        print(f"Word count avg:    {word_counts.mean():.1f}")
        print(f"Word count min/max:{word_counts.min()} / {word_counts.max()}")
    print("Completed label counts:")
    print(label_counts)
    print("═" * 60)

    part_deduped_path = Path(OUTPUT_PATH).with_name(f"cot_traces_{PART_NAME}_deduped.csv")
    df.to_csv(part_deduped_path, index=False)
    print(f"Part deduped copy saved to: {part_deduped_path}")
    return df

validated_df = validate_output()


In [ ]:
# ── Cell 11: Preview one generated trace per label ────────────────────────
rationale_col = "cot_rationale_esnli_style" if "cot_rationale_esnli_style" in validated_df.columns else "cot_rationale"
completed = validated_df[validated_df[rationale_col].fillna("").astype(str).str.strip() != ""]
if completed.empty:
    print("No completed generations yet.")
else:
    for _, row in completed.groupby("gold_label").head(1).iterrows():
        print(f"Label:      {row['gold_label']}")
        print(f"Premise:    {row['Sentence1']}")
        print(f"Hypothesis: {row['Sentence2']}")
        print(f"Human:      {row['Explanation_1']}")
        print(f"Generated:  {row[rationale_col]}")
        print("─" * 70)


In [ ]:
# ── Cell 12: Concatenate Part 1 + Part 2 into final cot_traces.csv ─────────
# Run this after Part 1 and Part 2 have both generated their assigned rows.

def concatenate_parts() -> pd.DataFrame:
    missing = [p for p in [PART1_OUTPUT_PATH, PART2_OUTPUT_PATH] if not Path(p).exists()]
    if missing:
        print("Missing part output(s). Finish both notebooks before concatenation:")
        for p in missing:
            print(" -", p)
        return pd.DataFrame()

    part1 = pd.read_csv(PART1_OUTPUT_PATH, dtype={"pair_id": int})
    part2 = pd.read_csv(PART2_OUTPUT_PATH, dtype={"pair_id": int})
    final = pd.concat([part1, part2], ignore_index=True)
    rationale_col = "cot_rationale_esnli_style" if "cot_rationale_esnli_style" in final.columns else "cot_rationale"

    final = final.drop_duplicates(subset=["pair_id"], keep="last").sort_values("pair_id").reset_index(drop=True)
    final.to_csv(FINAL_DEDUPED_PATH, index=False)

    completed = final[final[rationale_col].fillna("").astype(str).str.strip() != ""]
    completed.to_csv(FINAL_OUTPUT_PATH, index=False)

    print("═" * 60)
    print("Final concatenation report")
    print("═" * 60)
    print(f"Part 1 rows:       {len(part1):,}")
    print(f"Part 2 rows:       {len(part2):,}")
    print(f"Unique final rows: {len(final):,}")
    print(f"Completed rows:    {len(completed):,} / {TOTAL_TRACES:,}")
    print(f"Final output:      {FINAL_OUTPUT_PATH}")
    print(f"Final deduped:     {FINAL_DEDUPED_PATH}")
    print("Label counts:")
    print(completed["gold_label"].value_counts().sort_index())
    print("Available Model D columns:")
    for col in ["model_d_human_text", "model_d_cot_text", "model_d_blend_text"]:
        print(f" - {col}: {'yes' if col in completed.columns else 'missing'}")
    print("═" * 60)

    return completed

final_df = concatenate_parts()
